# Vector Databases — e-commerce product search, Pinecone + OpenAI embeddings

**Where we're headed.** By the end of this notebook, this line —

```python
index.query(vector=embed("wireless noise-cancelling headphones"), top_k=3)
```

— returns the **Sony WH-1000XM5**, **Bose QuietComfort 45**, and similar products from a 751-product catalog, even though none of their listings contain the phrase "noise-cancelling headphones" verbatim. That's the whole point of this notebook: the system finds *meaning*, not *keywords*.

Everything below builds up to that call, then goes past it into the operations a real system actually needs — updating, deleting, partitioning, and handling documents too long to embed in one piece.

Requires `OPENAI_API_KEY` and `PINECONE_API_KEY` set in your environment.

## 1 · The idea, in one line

> A **vector** is a list of numbers that carries the *meaning* of a sentence or word. We get that list from an **embedding model**. Similar meaning → similar numbers → nearby vectors — and finding nearby vectors is something a computer does very fast.

The score you'll see in results is a number from **−1 to +1**: closer to +1 means closer in meaning. You don't need to know how it's computed to use it correctly.

## 2 · The shape of a stored record

Every record Pinecone stores has three parts:

| Part | What it is | Example |
|---|---|---|
| `id` | a unique handle for the item | `"d2559c95-...-538c34a25bcb"` |
| `values` | the embedding — meaning, as coordinates | `[0.0123, -0.0456, ...]` (1536 numbers) |
| `metadata` | ordinary structured fields you filter on later | `{"price": 600.93, "currency": "USD"}` |

**Vectors find candidates. Metadata enforces rules.** Keep that split in mind — it's the pattern behind everything from here on.

In [3]:

from dotenv import load_dotenv

load_dotenv()

True

## 3 · Set up the two clients

In [4]:
# pip install openai pinecone pandas

import os
import pandas as pd
from openai import OpenAI      # turns text into vectors (the embedding model)
from pinecone import Pinecone, ServerlessSpec   # stores vectors and searches them

oa = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

EMBED_MODEL = "text-embedding-3-small"   # 1536 dims — must be the SAME model for every vector in this index
INDEX_NAME = "vidyasankalp-ecom-products"

def embed_batch(texts):
    """Turn a list of strings into a list of 1536-number vectors, one API call for the whole batch."""
    response = oa.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in response.data]   # same order as `texts`


## 4 · Load the dataset

**Dataset:** [kernel-memory-ecommerce-sample](https://github.com/yuniko-software/kernel-memory-ecommerce-sample) — 751 real product listings: `Name`, `Description`, `Price`, `PriceCurrency`, `SupplyAbility`, `MinimumOrder`.

In [6]:
DATA_URL = "products.csv"

df = pd.read_csv(DATA_URL, sep="|",)          # pipe-delimited, not comma
df

,Id,Name,Description,Price,PriceCurrency,SupplyAbility,MinimumOrder
0,d2559c95-bd28-49d8-b53a-538c34a25bcb,Saucony Men's Kinvara 13 Running Shoe,"When it comes to lightweight speed, nothing cr...",600.93,USD,396,574
1,c0717df8-fd27-4355-9c06-a8ad1ebb3e95,Accutire MS-4021B Digital Tire Pressure Gauge ...,About this item Heavy duty construction and ru...,476.98,USD,288,993
2,29ea5156-8582-4345-8672-65a8f4c1e30c,SAURA LIFE SCIENCE Adivasi Ayurvedic Neelgiri ...,This extraordinary fusion is designed to nouri...,216.74,EUR,640,707
3,cb17beb7-30f0-4857-8793-4a6df0983fa3,"crysting 13 Inch Sewing Box Three Layers, Plas...",About this item 🌟 High Quality Sewing Box 🌟 Ma...,84.40,EUR,85,923
4,b2d75b67-64f3-4ecd-82e6-47ff8e934c37,"Ridgid 62990 T-201 5"" Straight Auger","The product is 62990 AUGER, T201 STRAIGHT 5/8 ...",219.11,EUR,350,120
...,...,...,...,...,...,...,...
746,fc28844f-d741-4ac5-9fc2-d268b17810f4,Whirlpool 28 cu. ft. French Door Refrigerator,The Whirlpool 28 cu. ft. French Door Refrigera...,2199.99,USD,800,2
747,acfdc81f-06be-4c4e-a094-1cc35a24c227,VELCRO Brand - 91302 Thin Clear Dots with Adhe...,VELCRO Brand Thin Clear fasteners blend into g...,356.11,EUR,514,710
748,5ac4fd2c-0681-4de8-aaf4-dd8684436039,Madison Park Amherst Faux Silk Comforter Set-C...,The Madison Park Amherst 7 Piece Comforter Set...,383.20,EUR,831,960
749,d5e8517e-ee02-4d1d-bf1d-f7590e48a0d6,"BIC Gel-Ocity Gel Pens, Medium Point Retractab...",Gel-ocity is a smooth writing retractable gel ...,420.91,EUR,950,503


In [7]:
df["Description"] = df["Description"].fillna("")   # a couple of rows have no description

print(df.shape)
df.head()

(751, 7)


,Id,Name,Description,Price,PriceCurrency,SupplyAbility,MinimumOrder
0,d2559c95-bd28-49d8-b53a-538c34a25bcb,Saucony Men's Kinvara 13 Running Shoe,"When it comes to lightweight speed, nothing cr...",600.93,USD,396,574
1,c0717df8-fd27-4355-9c06-a8ad1ebb3e95,Accutire MS-4021B Digital Tire Pressure Gauge ...,About this item Heavy duty construction and ru...,476.98,USD,288,993
2,29ea5156-8582-4345-8672-65a8f4c1e30c,SAURA LIFE SCIENCE Adivasi Ayurvedic Neelgiri ...,This extraordinary fusion is designed to nouri...,216.74,EUR,640,707
3,cb17beb7-30f0-4857-8793-4a6df0983fa3,"crysting 13 Inch Sewing Box Three Layers, Plas...",About this item 🌟 High Quality Sewing Box 🌟 Ma...,84.40,EUR,85,923
4,b2d75b67-64f3-4ecd-82e6-47ff8e934c37,"Ridgid 62990 T-201 5"" Straight Auger","The product is 62990 AUGER, T201 STRAIGHT 5/8 ...",219.11,EUR,350,120


## 5 · Decide what gets embedded, and what stays metadata

We embed `Name + Description` — that's where the meaning lives. `Price`, `PriceCurrency`, `SupplyAbility`, `MinimumOrder` become metadata: facts *about* the product, used for filtering, never for search.

In [8]:
df["text"] = df["Name"] + ": " + df["Description"]
df.head(3)


,Id,Name,Description,Price,PriceCurrency,SupplyAbility,MinimumOrder,text
0,d2559c95-bd28-49d8-b53a-538c34a25bcb,Saucony Men's Kinvara 13 Running Shoe,"When it comes to lightweight speed, nothing cr...",600.93,USD,396,574,Saucony Men's Kinvara 13 Running Shoe: When it...
1,c0717df8-fd27-4355-9c06-a8ad1ebb3e95,Accutire MS-4021B Digital Tire Pressure Gauge ...,About this item Heavy duty construction and ru...,476.98,USD,288,993,Accutire MS-4021B Digital Tire Pressure Gauge ...
2,29ea5156-8582-4345-8672-65a8f4c1e30c,SAURA LIFE SCIENCE Adivasi Ayurvedic Neelgiri ...,This extraordinary fusion is designed to nouri...,216.74,EUR,640,707,SAURA LIFE SCIENCE Adivasi Ayurvedic Neelgiri ...


## 6 · Create the index (a one-time setup step)

In [10]:
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,          # must match EMBED_MODEL's output size exactly
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(INDEX_NAME)


## 7 · Embed and upsert all 751 products

Batch the embedding calls (one API call for many texts) and batch the upserts (Pinecone recommends ~100 records per call) — both standard practice at real scale.

In [16]:
def to_metadata(row):
    return {
        "text": row["text"],
        "price": float(row["Price"]),
        "currency": row["PriceCurrency"],
        "supply_ability": int(row["SupplyAbility"]),
        "minimum_order": int(row["MinimumOrder"]),
    }

In [20]:
BATCH = 100
for start in range(0, len(df), BATCH):
    print(start, start + BATCH)
    chunk_df = df.iloc[start:start + BATCH]
    text_list = chunk_df["text"].tolist()
    vectors = embed_batch(text_list)
    ids = chunk_df["Id"].tolist()
    metadata_rows = [to_metadata(row) for (row_id,row) in chunk_df.iterrows()]
    batch = [{"id":id, 'values':vector, "metadata":metadata} for id, vector, metadata in zip(ids, vectors, metadata_rows)]
    index.upsert(vectors=batch)
    print("-------------------")

0 100
-------------------
100 200
-------------------
200 300
-------------------
300 400
-------------------
400 500
-------------------
500 600
-------------------
600 700
-------------------
700 800
-------------------


In [ ]:
def to_metadata(row):
    return {
        "text": row["text"],
        "price": float(row["Price"]),
        "currency": row["PriceCurrency"],
        "supply_ability": int(row["SupplyAbility"]),
        "minimum_order": int(row["MinimumOrder"]),
    }

BATCH = 100
for start in range(0, len(df), BATCH):
    chunk = df.iloc[start:start + BATCH]
    vectors = embed_batch(chunk["text"].tolist())
    index.upsert(vectors=[
        {"id": row["Id"], "values": vec, "metadata": to_metadata(row)}
        for (_, row), vec in zip(chunk.iterrows(), vectors)
    ])

print(index.describe_index_stats())   # total vector count should read 751


## 8 · The payoff, for real this time

Note the query text shares almost no words with the winning product's listing — the match happens on meaning, not spelling.

In [21]:
query_vector = embed_batch(["wireless noise-cancelling headphones"])[0]   # embed the QUERY with the SAME model as the data

print(query_vector)

result = index.query(vector=query_vector, top_k=5, include_metadata=True)

for match in result["matches"]:
    m = match["metadata"]
    print(f'{match["score"]:.3f}  {m["text"][:70]:70s}  {m["price"]:>8.2f} {m["currency"]}')


[0.00901031494140625, -0.03375244140625, -0.05047607421875, -0.046356201171875, -0.035186767578125, -0.04205322265625, 0.00865936279296875, 0.062225341796875, 0.00847625732421875, -0.03076171875, -0.027191162109375, 0.044342041015625, -0.0589599609375, 0.0401611328125, -0.0185699462890625, -0.014923095703125, -0.0228424072265625, -0.0283355712890625, -0.01515960693359375, 0.0247955322265625, 0.005496978759765625, 0.07672119140625, 0.0148468017578125, 0.01529693603515625, 0.0192413330078125, 0.00615692138671875, 0.0261383056640625, 0.022796630859375, 0.050811767578125, 0.04425048828125, -0.039459228515625, -0.0145721435546875, 0.007904052734375, -0.020965576171875, -0.022705078125, -0.040008544921875, -0.00518035888671875, -0.039215087890625, -0.0094451904296875, -0.0002753734588623047, 0.0137786865234375, 0.0194854736328125, 0.01629638671875, 0.0120086669921875, 0.0168609619140625, 0.0188140869140625, -0.03167724609375, 0.00814056396484375, 0.01200103759765625, -0.005924224853515625, 0

## 9 · Update and delete — keeping the index honest

`upsert` isn't just for adding new records. Every operation an index actually needs in production comes down to three facts:

- **Upsert with an existing `id` overwrites that record.** No error, no duplicate — this is how you re-index a product whose price or description changed. Same call as §7, just called again on an id that already exists.
- **`index.delete(ids=[...])`** removes specific records by id — e.g. a product that's been discontinued.
- **`index.delete(filter={...})`** removes every record matching a metadata condition — e.g. wiping an entire stale batch at once. This uses the exact same filter syntax as querying (§10), which is worth noticing: filters mean the same thing whether you're searching or deleting.

We'll demonstrate on two throwaway records, tagged `"demo": True` in their metadata, so nothing below touches the real 751-product catalog.

In [ ]:
# --- create two small demo records, clearly marked so they never get confused with real data ---
demo_vectors = embed_batch([
    "Demo Product A: a placeholder item used only to demonstrate update and delete.",
    "Demo Product B: a placeholder item used only to demonstrate update and delete.",
])

index.upsert(vectors=[
    {"id": "demo-1", "values": demo_vectors[0], "metadata": {"text": "Demo Product A", "price": 10.0, "demo": True}},
    {"id": "demo-2", "values": demo_vectors[1], "metadata": {"text": "Demo Product B", "price": 20.0, "demo": True}},
])

print(index.fetch(ids=["demo-1"])["vectors"]["demo-1"]["metadata"])   # {'text': 'Demo Product A', 'price': 10.0, 'demo': True}


In [ ]:
# --- UPDATE: upserting the same id again overwrites it in place, no duplicate is created ---
updated_vector = embed_batch(["Demo Product A: now on clearance, price reduced."])[0]

index.upsert(vectors=[
    {"id": "demo-1", "values": updated_vector, "metadata": {"text": "Demo Product A (clearance)", "price": 4.99, "demo": True}},
])

print(index.fetch(ids=["demo-1"])["vectors"]["demo-1"]["metadata"])   # price is now 4.99 — same id, updated content


In [ ]:
# --- DELETE by id: remove one specific record ---
index.delete(ids=["demo-2"])

try:
    index.fetch(ids=["demo-2"])["vectors"]["demo-2"]
    print("still present (unexpected)")
except KeyError:
    print("demo-2 is gone")


In [ ]:
# --- DELETE by filter: remove everything matching a metadata condition, in one call ---
# scoped to demo:True so this can never accidentally touch the real catalog
index.delete(filter={"demo": {"$eq": True}})

print(index.describe_index_stats())   # total count is back to 751 — both demo records are gone


## 10 · Metadata filtering — worth its own section

Real search is rarely just "closest vector." It's "closest vector that also costs under $500 and is well-stocked." This is the same **vectors find candidates, metadata enforces rules** split from §2 — and it's a pattern you'll reuse constantly: access control (filter by tenant), freshness (filter by date), inventory (filter by stock), permissions (filter by role).

**The operators available:**

| Operator | Meaning | Example |
|---|---|---|
| `$eq` | equals | `{"currency": {"$eq": "USD"}}` |
| `$ne` | not equal | `{"currency": {"$ne": "EUR"}}` |
| `$gt` / `$gte` | greater than / or equal | `{"price": {"$gte": 100}}` |
| `$lt` / `$lte` | less than / or equal | `{"price": {"$lt": 500}}` |
| `$in` | value is one of a list | `{"currency": {"$in": ["USD", "EUR"]}}` |
| `$nin` | value is none of a list | `{"currency": {"$nin": ["JPY"]}}` |

**Combining conditions:** multiple keys in one `filter={}` dict are **AND**ed together automatically. For **OR** logic, wrap conditions explicitly with `$or`.

Two pitfalls: **type must match exactly** (filtering `price` with a string instead of a float silently matches nothing, not an error), and **you can only filter on fields stored as metadata at upsert time** — there's no adding a filterable field after the fact without re-upserting.

In [ ]:
# Example 1 — AND: currency is USD, price is under 500, and supply_ability is at least 1000
result = index.query(
    vector=embed_batch(["a laptop for creative professional work"])[0],
    top_k=5,
    include_metadata=True,
    filter={
        "currency": {"$eq": "USD"},
        "price": {"$lt": 500},
        "supply_ability": {"$gte": 1000},
    },
)

for match in result["matches"]:
    m = match["metadata"]
    print(f'{match["score"]:.3f}  {m["text"][:70]:70s}  {m["price"]:>8.2f} {m["currency"]}')


**Example 2 — `$in`:** products priced in either currency, but only the cheap end of the catalog.

In [ ]:
result = index.query(
    vector=embed_batch(["a sturdy gift for someone who works outdoors"])[0],
    top_k=5,
    include_metadata=True,
    filter={
        "currency": {"$in": ["USD", "EUR"]},
        "price": {"$lt": 100},
    },
)

for match in result["matches"]:
    m = match["metadata"]
    print(f'{match["score"]:.3f}  {m["text"][:70]:70s}  {m["price"]:>8.2f} {m["currency"]}')


**Example 3 — explicit `$or`:** either a cheap USD item, or any item with very high supply (bulk-order friendly), regardless of price.

In [ ]:
result = index.query(
    vector=embed_batch(["a reliable everyday backpack"])[0],
    top_k=5,
    include_metadata=True,
    filter={
        "$or": [
            {"currency": {"$eq": "USD"}, "price": {"$lt": 100}},
            {"supply_ability": {"$gte": 5000}},
        ]
    },
)

for match in result["matches"]:
    m = match["metadata"]
    print(f'{match["score"]:.3f}  {m["text"][:70]:70s}  {m["price"]:>8.2f} {m["currency"]}  (supply {m["supply_ability"]})')


## 11 · Namespaces — partitioning one index

A **namespace** splits one index into isolated sub-spaces. A query only ever searches within the namespace you name — records in other namespaces are invisible to it, with zero extra infrastructure. The standard uses:

- **Multi-tenant systems** — each customer's data in its own namespace, so tenant A can never see tenant B's vectors, even by accident.
- **Environment separation** — `dev` and `prod` namespaces in the same index, instead of paying for two indexes.
- **Separate datasets sharing one index** — e.g. a product catalog namespace and a support-docs namespace, if both happen to use the same embedding model.

This solves the same problem your ACT platform solves with a single-table DynamoDB design and typed sort keys — different mechanism (physical partitioning vs. key structure), same goal: keep tenants' data cleanly separated without multiplying infrastructure.

We'll demonstrate by splitting a small slice of the catalog into two namespaces by currency — a stand-in for "US storefront" vs. "EU storefront".

In [ ]:
usd_sample = df[df["PriceCurrency"] == "USD"].head(10)
eur_sample = df[df["PriceCurrency"] == "EUR"].head(10)

for sample, ns in [(usd_sample, "storefront-us"), (eur_sample, "storefront-eu")]:
    vectors = embed_batch(sample["text"].tolist())
    index.upsert(
        vectors=[
            {"id": row["Id"], "values": vec, "metadata": to_metadata(row)}
            for (_, row), vec in zip(sample.iterrows(), vectors)
        ],
        namespace=ns,   # <-- the only difference from a normal upsert
    )

print(index.describe_index_stats())   # namespaces breakdown shows storefront-us: 10, storefront-eu: 10, '' (default): 751


In [ ]:
# Querying is scoped to one namespace at a time — results from other namespaces never appear
result = index.query(
    vector=embed_batch(["a comfortable running shoe"])[0],
    top_k=3,
    include_metadata=True,
    namespace="storefront-us",   # <-- only searches the 10 records upserted above under this namespace
)

for match in result["matches"]:
    m = match["metadata"]
    print(f'{match["score"]:.3f}  {m["text"][:70]:70s}  {m["price"]:>8.2f} {m["currency"]}')

# Omitting `namespace=` (as in every query earlier in this notebook) searches the default namespace, "" —
# which is where all 751 products from §7 live. storefront-us and storefront-eu are additive, not a split.


## 12 · Chunking long text before embedding

Every embedding model has a maximum input length. `text-embedding-3-small` accepts up to 8,191 tokens (~6,000 words) — generous, but real documents blow past it: a 40-page PDF, a legal filing, a full product manual. Text beyond the limit isn't rejected — it's **silently truncated**, and the tail of the document never influences the vector at all. This is one of the most common real-world bugs in retrieval systems, and it produces no error message.

Our product descriptions were short enough to dodge this entirely. Here's the fix for when they aren't: split long text into overlapping chunks *before* embedding, embed each chunk separately, and store each as its own vector — with metadata linking it back to the source document.

In [ ]:
def chunk_text(text, chunk_size=60, overlap=15):
    """Split text into overlapping word-count chunks.
    chunk_size=60, overlap=15 are small on purpose, to make the effect visible in this demo —
    real settings are usually a few hundred words with 10-20% overlap.
    The overlap matters: without it, a sentence split across a chunk boundary can lose its meaning
    in both halves. With it, the sentence survives intact in at least one chunk.
    """
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap   # step forward by less than chunk_size, so chunks overlap
    return chunks

# a synthetic "long document" — several product descriptions concatenated, standing in for
# something like a 40-page manual that would otherwise exceed the embedding model's input limit
long_document = " ".join(df["Description"].head(15).tolist())
print(f"{len(long_document.split())} words total")

chunks = chunk_text(long_document)
print(f"split into {len(chunks)} chunks")
print(chunks[0][:200], "...")


In [ ]:
# Embed and upsert each chunk as its own vector — metadata links every chunk back to its source
# document and records its position, so a matching chunk can be traced back to where it came from.
chunk_vectors = embed_batch(chunks)

index.upsert(vectors=[
    {
        "id": f"demo-doc-1-chunk-{i}",
        "values": vec,
        "metadata": {"text": chunk, "source_doc": "demo-doc-1", "chunk_index": i, "demo": True},
    }
    for i, (chunk, vec) in enumerate(zip(chunks, chunk_vectors))
])

# query against the chunks specifically, using the same demo:True tag so this stays isolated
result = index.query(
    vector=embed_batch(["information about battery life and charging"])[0],
    top_k=2,
    include_metadata=True,
    filter={"source_doc": {"$eq": "demo-doc-1"}},
)

for match in result["matches"]:
    m = match["metadata"]
    print(f'{match["score"]:.3f}  chunk {m["chunk_index"]}: {m["text"][:100]}...')

# clean up the demo chunks — same delete-by-filter pattern from §9
index.delete(filter={"demo": {"$eq": True}})


## 13 · Why this works — cosine similarity, then IVF and HNSW

Everything above ran without ever needing this section. It's here because "why does this stay fast, and what is that score actually measuring" are the two questions worth answering once you've seen the system work.

### 13.1 · The score — cosine similarity, worked out

Every `match["score"]` you printed above is **cosine similarity**: a number from **−1 to +1** measuring how closely two vectors point in the same direction.

$$\text{cosine}(a, b) = \frac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}$$

Three landmarks are all you need to read it: **+1** = pointing the same way (same meaning), **0** = unrelated, **−1** = pointing opposite ways. In practice, real product matches in this notebook scored roughly 0.2–0.4 — cosine scores for real short text rarely get close to 1, because most of a 1536-dimension embedding captures generic "this is English text about a product" structure that every listing shares; only a slice of those dimensions carries the specific meaning that separates one product from another. **Relative ranking, not the absolute number, is what matters** — the point isn't "is 0.31 a big number," it's "is 0.31 bigger than the next result's score."

Two vectors get compared with the dot product, then divided by their lengths — dividing by length is what makes it *cosine* similarity rather than just "dot product," and it's why length (how much text there is) doesn't skew the score, only direction (what the text is about) does.

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

# grab two real vectors already sitting in Pinecone, to see an actual score computed by hand
fetched = index.fetch(ids=[df.iloc[0]["Id"], df.iloc[1]["Id"]])["vectors"]
v0 = fetched[df.iloc[0]["Id"]]["values"]
v1 = fetched[df.iloc[1]["Id"]]["values"]

print(df.iloc[0]["Name"], "vs.", df.iloc[1]["Name"])
print("cosine similarity:", round(cosine_similarity(v0, v1), 4))
print("(Pinecone computed the exact same number internally on every query() call above —")
print(" it just does it against all 751 vectors at once, and fast.)")


### 13.2 · Why comparing against every vector doesn't scale

`index.query()` above never scans all 751 vectors one by one and computes `cosine_similarity` 751 times, even though that's what §13.1 just did by hand for one pair. Comparing a query against *every* stored vector is called **brute-force (flat) search** — exact, simple, and the one thing that stops being affordable once an index holds millions of vectors instead of hundreds. Double the catalog, double every query's cost, forever.

This is where **ANN (Approximate Nearest Neighbour)** search comes in: return results that are almost always the true nearest ones, in exchange for not having to touch every vector. Two index designs solve this, and it's worth knowing both by name, because you'll see them in every vector database's documentation.

### 13.3 · IVF — cluster first, then search only the right cluster

**IVF (Inverted File Index)** groups all stored vectors into clusters at build time, and records each cluster's centre point. A query first compares itself only against the (much smaller) set of cluster centres, then searches exhaustively *inside* just the nearest one or two clusters — skipping every vector in every other cluster entirely.

Think of it like sorting mail by postal code: to deliver one letter, you don't check every house in the country — you jump straight to its postal code and search only there. Fast, and almost always correct. The one failure mode: an item that sits right on the boundary between two clusters can occasionally get filed on the "wrong" side and missed.

### 13.4 · HNSW — walk a graph toward the answer

**HNSW (Hierarchical Navigable Small World)** is the more common approach today, and what Pinecone's serverless indexes use under the hood. Instead of clusters, it builds a multi-layer graph of "who is near whom" — sparse long-range links on top for covering large distances quickly, denser short-range links below for fine adjustment.

A query starts somewhere in the top layer and repeatedly hops to whichever neighbour is closer to the query, dropping down a layer each time it stops improving. Picture asking for directions in an unfamiliar town: you don't knock on every door — you ask a local, get pointed a couple of streets closer, ask again, and in four or five hops you're at the right door. You never visited most of the town, and you almost always arrive correctly.

Pinecone's serverless indexes (the kind `ServerlessSpec` created in §6) manage this entirely for you — there's no `index_type="hnsw"` parameter to set. Knowing it's HNSW underneath is useful for reasoning about behaviour (why results are *almost always* the true top-k, why very large indexes stay fast, why recall is tunable in self-managed engines like OpenSearch or Qdrant even though Pinecone doesn't expose the knob directly), not for configuring anything here.

### 13.5 · IVF vs. HNSW — and which is actually better

Quick recap of the names, since they're worth knowing cold: **IVF = Inverted File Index** (cluster first, search the nearest cluster). **HNSW = Hierarchical Navigable Small World** (hop through a proximity graph).

| | Flat (brute force) | IVF | HNSW |
|---|---|---|---|
| Method | compare against everything | cluster, then search nearest cluster(s) | hop through a proximity graph |
| Exact? | yes | no — approximate | no — approximate |
| Typical use | small collections (few thousand vectors) | large collections, tunable via `nprobe` | large collections; Pinecone's default |
| Failure mode | none — just slow at scale | boundary items occasionally missed | rare, correctable by searching more of the graph |

**Head-to-head, IVF vs. HNSW specifically:**

| | IVF | HNSW |
|---|---|---|
| Recall at a given speed | lower | higher |
| Query speed | fast | generally faster |
| Memory use | lower | higher — the graph itself takes real RAM |
| Update-friendliness | needs periodic re-clustering as data changes | handles inserts/updates more gracefully |
| Build time | faster to build | slower to build |
| Tuning knobs | one (`nprobe` — how many clusters to search) | two (`ef_construction`, `ef_search`) |

**So which is better?** HNSW, for most real workloads — it's why it's become the default in Pinecone, Weaviate, Qdrant, and pgvector. The honest exception: IVF still wins on **memory-constrained or very large-scale** deployments (billions of vectors), where HNSW's graph simply doesn't fit affordably in RAM — Milvus and FAISS-based systems often default to IVF, or a hybrid IVF+PQ that also compresses the vectors, for exactly that reason. Some engines also run IVF and HNSW together — IVF for coarse clustering, HNSW searched within each cluster.

For this notebook specifically, none of this is a decision you get to make: Pinecone's serverless index uses HNSW internally and doesn't expose the choice. This section is "know what's happening underneath," not "know which flag to set" — useful the day you're choosing a self-managed engine (OpenSearch, Qdrant, Milvus) where that flag exists.

Whichever index type is underneath, one thing never changes: **the score returned is still cosine similarity** (§13.1), and **query and data must still share one embedding model** (§8) — ANN changes *which* vectors get compared, never *how* two vectors are compared once they are.

Full treatment — the inverted index BM25 uses instead (and why it stays exact where dense search can't), plus real measured recall-vs-speed numbers — is Part VI of the companion book, and the BM25 notebook's §9.

## Recap

- `text` (name + description) is what gets embedded; everything else is metadata.
- `upsert` both creates and updates — same id in, overwritten in place. `delete(ids=[...])` and `delete(filter={...})` remove records, by id or in bulk.
- Metadata filtering (`$eq`, `$in`, `$or`, ...) is a reusable pattern: vectors find candidates, metadata enforces rules.
- Namespaces partition one index for tenants, environments, or datasets — a query only ever sees its own namespace.
- Long documents must be chunked with overlap before embedding, or the tail is silently dropped.

**Want the exact math** behind cosine similarity, the dot product, and BM25? That's Appendix A of the companion book — optional.